In [65]:
import pandas as pd
import sqlite3

# Loading Raw ACS Data into a DataFrame

In [66]:
df = pd.read_excel("../data/oo_raw.xlsx")

# Exploratory Data Analysis

Using a Custom Function from the Utilities Notebook

In [67]:
from utilities import basic_eda

basic_eda(df)

DataFrame Shape: (5245, 23)

 Column Names:
['Area Name', 'Area Type', 'SOC Title', 'Standard Occupational Classification (SOC)', 'SOC Major Group', 'SOC Classification', '2022 Estimated Employment', '2032 Projected Employment', 'Change', 'Percent Change', 'Annualized Percent Growth (Grow Rate)', 'Exits', 'Transfers', 'Openings', 'Mean Annual', 'Entry Annual', '25th Percentile Anual', 'Median Annual', '75th Percentile Annual', 'Experienced Annual', 'Typical Education Required for Entry', 'Typical Work Experience Required in Related Occupation', 'Typical On-the-Job Training Required to Achieve Competency']

 Data Types:
Area Name                                                      object
Area Type                                                      object
SOC Title                                                      object
Standard Occupational Classification (SOC)                      int64
SOC Major Group                                                 int64
SOC Classification     

# Cleaning

## Column Names

I will start by changing column names to descriptive ones in lower snake case.

In [68]:
# Create a column renaming dictionary
column_rename_map = {
    "Area Name": "area_name",
    "Area Type": "area_type",
    "SOC Title": "soc_title",
    "Standard Occupational Classification (SOC)": "occupation",
    "SOC Major Group": "soc_major_group",
    "SOC Classification": "soc_classification",
    "2022 Estimated Employment": "employment_2022",
    "2032 Projected Employment": "employment_2032",
    "Change": "employment_change",
    "Percent Change": "percent_change",
    "Annualized Percent Growth (Grow Rate)": "annualized_percent_growth",
    "Exits": "exits",
    "Transfers": "transfers",
    "Openings": "openings",
    "Mean Annual": "mean_annual_wage",
    "Entry Annual": "entry_annual_wage",
    "25th Percentile Anual": "percentile_25_wage",
    "Median Annual": "median_annual_wage",
    "75th Percentile Annual": "percentile_75_wage",
    "Experienced Annual": "experienced_annual_wage",
    "Typical Education Required for Entry": "education_required",
    "Typical Work Experience Required in Related Occupation": "work_experience_required",
    "Typical On-the-Job Training Required to Achieve Competency": "ojt_required"
}

# Apply the renaming
df = df.rename(columns=column_rename_map)

# Check the result
print(df.columns.tolist())


['area_name', 'area_type', 'soc_title', 'occupation', 'soc_major_group', 'soc_classification', 'employment_2022', 'employment_2032', 'employment_change', 'percent_change', 'annualized_percent_growth', 'exits', 'transfers', 'openings', 'mean_annual_wage', 'entry_annual_wage', 'percentile_25_wage', 'median_annual_wage', 'percentile_75_wage', 'experienced_annual_wage', 'education_required', 'work_experience_required', 'ojt_required']


## Handling Nulls

### Observations
1. The columns 'soc_classification', 'work_experience_required', and 'ojt_required' all have a large percentage of nulls.  
2. Several other columns have low percentages of nulls.

### Thoughts/Plan
1. Eliminate the 3 columns with nulls >10%. They are not necessary.
2. Fill nulls in the 'education_required' column to 'Not reported'.
3. Fill nulls in the 'occupation' column to 'Not employed.'
3. Leave other nulls in Dataframe. 
    - These columns have null percentages <10/%.
    - Using a mean or median fill value for these columns would be misleading.
    - Nulls can be filtered during analysis.

In [69]:
columns_to_drop = [
    "soc_classification",
    "work_experience_required",
    "ojt_required"
]

df = df.drop(columns=columns_to_drop) # Eliminating columns with high percentage of nulls

df["education_required"] = df["education_required"].fillna("Not reported")

#For later merging, I need this column to be a string.
df['occupation'] = df['occupation'].astype(str).str.zfill(6)
df["occupation"] = df["occupation"].fillna("Not employed")


## Rechecking Data Types

In [70]:
df.dtypes

area_name                     object
area_type                     object
soc_title                     object
occupation                    object
soc_major_group                int64
employment_2022                int64
employment_2032                int64
employment_change              int64
percent_change               float64
annualized_percent_growth    float64
exits                          int64
transfers                      int64
openings                       int64
mean_annual_wage             float64
entry_annual_wage            float64
percentile_25_wage           float64
median_annual_wage           float64
percentile_75_wage           float64
experienced_annual_wage      float64
education_required            object
dtype: object

# Converting to SQLite Database Table

In [71]:
# Connect to existing SQLite database
conn = sqlite3.connect("../data/cleaned_data.sqlite")

# Write the DataFrame `df` to the database as a new table called "puma_data"
df.to_sql(
    name="oo_data",       # name of the new table
    con=conn,               # database connection
    if_exists="replace",    # overwrite if table already exists
    index=False             # don't include the index as a column
)

# Close the connection
conn.close()

print("Table 'oo_data' successfully added to cleaned_data.sqlite.")

Table 'oo_data' successfully added to cleaned_data.sqlite.


In [77]:
df['occupation'].astype(str).str.pad(6, fillchar='0').unique()



array(['000000', '110000', '111011', '111021', '112011', '112021',
       '112022', '112032', '112033', '113012', '113013', '113021',
       '113031', '113051', '113061', '113071', '113111', '113121',
       '113131', '119013', '119031', '119032', '119033', '119039',
       '119041', '119051', '119071', '119081', '119111', '119121',
       '119131', '119141', '119151', '119161', '119171', '119199',
       '130000', '131011', '131020', '131031', '131032', '131041',
       '131051', '131071', '131075', '131081', '131082', '131111',
       '131121', '131131', '131141', '131151', '131161', '131199',
       '132011', '132031', '132041', '132051', '132052', '132054',
       '132061', '132071', '132072', '132081', '132082', '132099',
       '150000', '151211', '151212', '151221', '151231', '151232',
       '151241', '151242', '151243', '151244', '151251', '151252',
       '151253', '151254', '151255', '151299', '152031', '152041',
       '152099', '170000', '171011', '171012', '171021', '1710